In [1]:
reset()

In [2]:
import random

In [3]:
def bits(N):
    if N == 0:
        return 0
    else:
        return floor(log(N,2))+1

def pad_binary(bin_str):
    return '0' * (11 - len(bin_str)) + bin_str

def count_equal(L1, L2):
    return sum(x == y for x, y in zip(L1, L2))

def correction_of_msg(N,m):
    '''
    N: the parameter of NTRU
    m: the message polynomial
    Output: a list of length N with the coefficients of m(x)
    '''
    m_list=m.list()
    if len(m_list)<N:
        diff = N - len(m_list)
        m_list.append(diff*[0])
    return flatten(m_list)

def random32():
    return randrange(-2^31,2^31)        # a random integer in -2^31 and 2^31

def randomrange3():
    '''
    generate a random integer in {0, 1, 2}
    by utilizing a 32-bit random number generator : random32()
    '''
    return ((random32() & 0x3fffffff) * 3) >> 30

# we generate ternary polynomials of degree at most N with d1 one and d2 minus one
def T(d1, d2, N, Zx):
    a = d1*[1]
    b = d2*[-1]
    c = (N-d1-d2)*[0] # the length must be N+1 to get a polynomial of degree at most N
    L = flatten([a,b,c])
    random.shuffle(L)
    return L, Zx(L)

def centerLift(f,q,N, Zx):
    f_balanced = list(   ((f[i]+q//2)%q) -q//2  for i in range(N)   )
    return Zx(f_balanced)

def reduce_mod_DN_and_modq(f, Sq, Zx, R): # if f=G*F then we get the multiplication of G and F in R/(q,x^N-1)
    output = Sq(R(f).lift())
    return Zx(output)

def convolution_in_S_q(f,g,N,q, Zx):
    Qmod.<qModX> = PolynomialRing(Integers(q))
    output = convolution_in_S(f,g,N, Zx)
    return  Zx(Qmod(output))

def convolution_in_S(f,g,N, Zx):
    Zx.<x> = ZZ[]
    D = x^N - 1
    h = (f*g)%(D)
    return  Zx(h)

def invert_mod_power_of_prime_mod_DN(f, Q, e, N, Zx, S2): # Q prime, e: exponent
    F = invert_mod2_and_DN(f, Zx, S2)
    if e == 1:
        return F
    temp_exponent = 2
    while e > 0:
        temp = convolution_in_S(F,f,N, Zx)
        F = convolution_in_S_q(F,2-temp,N,Q^temp_exponent, Zx)
        e = floor(e/2)
        temp_exponent = 2 * temp_exponent
    return F

def invert_mod2_and_DN(f, Zx, S2):
    return Zx( (1/S2(f)).lift())

def invert_mod3_and_DN(f, Zx, S3):
    return Zx( (1/S3(f)).lift() )

def private_keys(N, d, Zx, S3):
    # for L_g we choose d=q/16-1
    # for L_f we choose T_{N-2}
    while True:
        f = Zx([randomrange3()-1 for i in range(N-1)])
        if S3(f).is_unit(): break
    g = T(d,d,N, Zx)
    return f, g[1]

def gen_keys(N, q, d, Zx, S3, Sq, S2, R):
    f, g = private_keys(N, d, Zx, S3)

    # Compute invert of f modulo p, f modulo q and h modulo q
    base_of_q = 2
    exponent  = int(log(q,base_of_q)) # q = (base_of_q)^(exponent)
    try:
        fq = invert_mod_power_of_prime_mod_DN(f, base_of_q, exponent, N, Zx, S2)
    except ZeroDivisionError:
        # print("Oops! there is no inverse of f in S_{0} (q)".format(q))
        return None,None,None,None,None

    try:
        f3 = invert_mod3_and_DN(f, Zx, S3)
    except ZeroDivisionError:
        # print("Oops! there is no inverse of f in S_{0} (p)".format(3))
        return None,None,None,None,None

    h = reduce_mod_DN_and_modq(fq * g, Sq, Zx, R) # public key, we work mod<q,x^N-1>
    return f, g, f3, fq, h

def encrypt(N, d, msg, h, Zx, Sq, R):
    r2 = Zx([randomrange3()-1 for i in range(N)]) #ephemeral key, a ternary of degree <=N-2
    r1 = [i for i in r2]
    if len(r1) < N:
        r1 = r1+ (N-len(r1))*[0]
    r = (r1,r2)

    h3 = 3*h
    ct1 = reduce_mod_DN_and_modq(h3*r[1], Sq, Zx, R)  # mod <q,x^N-1>
    ct = reduce_mod_DN_and_modq(ct1 + msg, Sq, Zx, R)   # mod <q,x^N-1>
    return ct,r

def decrypt(ct, f, f3, m, Zx, S3, Sq, R):
    a = reduce_mod_DN_and_modq(ct * f, Sq, Zx, R)
    a = centerLift(a,q,N, Zx)
    # if S3(a*f3) == S3(m):
    #     print("decryption OK")
    return Zx( S3(a*f3).lift() )

def gen_msg(N, d, Zx):
    msg = T(d, d, N, Zx)
    m = msg[1]
    while m.degree()==N-1:
        msg = T(d, d, N, Zx)
        m = msg[1]
    return msg